# Notebook 01 — Replicator vs Fermi update rule (L = 1)

**Goal:** Compare the replicator (proportional imitation) and Fermi update rules at L = 1,
where both duplex layers share the same network. This reproduces qualitatively the results
of Pereda (2016) *Phys. Rev. E* 94, 032314 and establishes Fermi as the update rule
for the rest of the paper.

Networks: BA and ER, z = 4 and z = 16.  
Parameters: b ∈ [1.0, 2.0], θ ∈ [0.0, 1.0], N = 1000, 100 replications.

In [ ]:
import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import model

In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────
N        = 1000
n_rep    = 100      # set to 10 for a quick smoke test
L        = 1
lam      = 0.5     # irrelevant at L=1 (alpha=[1.0] by normalization)
K_fermi  = 0.1
b_vals   = np.round(np.linspace(1.0, 2.0, 11), 2)
th_vals  = np.round(np.linspace(0.0, 1.0, 11), 2)
NET_SEED = 0       # one fixed network per topology
SIM_SEED = 0       # base seed for replications
N_JOBS   = -1      # use all cores

print('b:', b_vals)
print('θ:', th_vals)

In [ ]:
# ── Build networks and pre-compile JIT kernels ──────────────────────────────
G_BA4  = model.build_network('BA', N, 4,  seed=NET_SEED)
G_BA16 = model.build_network('BA', N, 16, seed=NET_SEED)
G_ER4  = model.build_network('ER', N, 4,  seed=NET_SEED)
G_ER16 = model.build_network('ER', N, 16, seed=NET_SEED)

NETWORKS = [
    ('BA_z4',  G_BA4,  'BA, z = 4'),
    ('BA_z16', G_BA16, 'BA, z = 16'),
    ('ER_z4',  G_ER4,  'ER, z = 4'),
    ('ER_z16', G_ER16, 'ER, z = 16'),
]

for key, G, title in NETWORKS:
    n = G.number_of_nodes()
    k = 2 * G.number_of_edges() / n
    d = nx.diameter(G)
    print(f"{title:12s}  N={n}  <k>={k:.2f}  diameter={d}")

print()
model.warm_up()
print('JIT kernels compiled.')

In [ ]:
# ── Helper: cached sweep ───────────────────────────────────────────────────
os.makedirs('data', exist_ok=True)

def cached_sweep(G, key, update_rule):
    """Run or load a (b, θ) sweep. Results cached to data/."""
    fname = f"data/01-{key}-{update_rule}.csv"
    if os.path.exists(fname):
        print(f"  loading {fname}")
        return pd.read_csv(fname)
    gp, gd = model.game_csr(G)
    sp, sd = model.shells_csr(G, L)
    al     = model.geometric_kernel(L, lam)
    df = model.run_sweep(
        gp, gd, sp, sd, al,
        b_vals, th_vals,
        K=K_fermi, n_rep=n_rep,
        n_jobs=N_JOBS, base_seed=SIM_SEED,
        update_rule=update_rule,
    )
    df.to_csv(fname, index=False)
    print(f"  saved {fname}")
    return df

In [ ]:
# ── Run sweeps (or load from cache) ────────────────────────────────────────
results = {}
for key, G, title in NETWORKS:
    for rule in ('rep', 'fermi'):
        print(f"{title} — {rule} ...", flush=True)
        results[(key, rule)] = cached_sweep(G, key, rule)
print('Done.')

In [ ]:
# ── Heatmaps: ⟨ρ⟩(b, θ) for all networks and both rules ────────────────────
def rho_matrix(df):
    """Return 11×11 array: rows = θ ascending (bottom→top), cols = b ascending."""
    return (
        df.pivot(index='theta', columns='b', values='rho_mean')
          .sort_index(ascending=True)
          .values
    )

ext = [b_vals[0] - 0.05, b_vals[-1] + 0.05, th_vals[0] - 0.05, th_vals[-1] + 0.05]
rules = [('rep', 'Replicator (PRE 2016)'), ('fermi', 'Fermi (K = 0.1)')]

fig, axes = plt.subplots(
    len(NETWORKS), len(rules),
    figsize=(7.5, 13),
    constrained_layout=True,
)

for row, (key, G, title) in enumerate(NETWORKS):
    for col, (rule, rule_label) in enumerate(rules):
        ax  = axes[row, col]
        mat = rho_matrix(results[(key, rule)])
        im  = ax.imshow(
            mat, origin='lower', aspect='auto',
            vmin=0, vmax=1, cmap='Blues', extent=ext,
        )
        ax.set_xticks([1.0, 1.25, 1.5, 1.75, 2.0])
        ax.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
        ax.set_xlabel('b', fontsize=10)
        ax.set_ylabel('θ', fontsize=10)
        if row == 0:
            ax.set_title(rule_label, fontsize=11, fontweight='bold')
        ax.text(
            0.03, 0.97, title, transform=ax.transAxes,
            fontsize=9, va='top', color='white',
            bbox=dict(facecolor='black', alpha=0.35, pad=2, edgecolor='none'),
        )
        cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
        cb.set_label('⟨ρ⟩', fontsize=9)
        cb.set_ticks([0, 0.25, 0.5, 0.75, 1.0])

fig.suptitle('Stationary cooperator fraction — L = 1 (correlated duplex)',
             fontsize=12, y=1.01)

os.makedirs('figures', exist_ok=True)
fig.savefig('figures/01-REP-vs-Fermi.pdf', bbox_inches='tight')
fig.savefig('figures/01-REP-vs-Fermi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/01-REP-vs-Fermi.{pdf,png}')

In [ ]:
# ── Marginal: ⟨ρ⟩ vs b at fixed θ ──────────────────────────────────────────
theta_target = 0.5

fig2, axes2 = plt.subplots(2, 2, figsize=(9, 7), constrained_layout=True)
axes2 = axes2.flatten()

for idx, (key, G, title) in enumerate(NETWORKS):
    ax = axes2[idx]
    for rule, rule_label, ls, color in [
        ('rep',   'Replicator', '-',  'C0'),
        ('fermi', 'Fermi',      '--', 'C1'),
    ]:
        df  = results[(key, rule)]
        sub = df[np.isclose(df['theta'], theta_target)].sort_values('b')
        ax.plot(sub['b'], sub['rho_mean'], ls, label=rule_label, lw=2, color=color)
        ax.fill_between(
            sub['b'],
            (sub['rho_mean'] - sub['rho_std']).clip(0),
            (sub['rho_mean'] + sub['rho_std']).clip(0, 1),
            alpha=0.15, color=color,
        )
    ax.set_xlim(1.0, 2.0)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('b', fontsize=10)
    ax.set_ylabel('⟨ρ⟩', fontsize=10)
    ax.set_title(f'{title}  (θ = {theta_target})', fontsize=10)
    ax.legend(fontsize=9)
    ax.axhline(0, color='grey', lw=0.5, ls=':')
    ax.axhline(1, color='grey', lw=0.5, ls=':')

fig2.suptitle(f'⟨ρ⟩ vs b at θ = {theta_target} — REP vs Fermi, L = 1', fontsize=12)
fig2.savefig('figures/01-marginal-theta05.pdf', bbox_inches='tight')
fig2.savefig('figures/01-marginal-theta05.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/01-marginal-theta05.{pdf,png}')

## Observations

*(Fill in after running)*

- REP and Fermi at L = 1: qualitative agreement / differences?
- Does Fermi reproduce the PRE 2016 cooperation-enhancing effect of vigilance?
- Differences between BA and ER, z = 4 vs z = 16?
- Justification for using Fermi for L > 1: avoids Φ normalization with heterogeneous T_i.